<a href="https://colab.research.google.com/github/KayraOrbay/GROUP37_CENG313/blob/eda-guncelleme/assignment1_eda_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CENG313 PROJECT

Problem Definition


This project is based on the competion in the Kaggle *Image Matching Challenge 2025*. Our main goal is to estimate 3D camera poses by identifying reliable matching points between 2D images of the same scene, which are captured from various viewpoints and lighting conditions.

Instead of a traditional tabular target, the dataset targets rotation matrices and translation vectors between images. These matrices will be calculated using matched keypoints.


The model's performance is measured using the mAA (Mean Average Accuracy) metric. This metric evaluates the accuracy of our predicted camera poses by calculating the rotation and translation errors compared to the ground truth.

In [11]:
print("Colab çalışıyor")

Colab çalışıyor


In [12]:
%pip install -q uv

In [13]:
!git clone https://github.com/KayraOrbay/GROUP37_CENG313.git

Cloning into 'GROUP37_CENG313'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 36 (delta 14), reused 11 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 16.69 KiB | 416.00 KiB/s, done.
Resolving deltas: 100% (14/14), done.


In [14]:
%cd /content/GROUP37_CENG313
!ls

/content/GROUP37_CENG313
assignment1_eda_baseline.ipynb	GROUP37_CENG313  requirements.txt
data				README.md	 src


In [15]:
import subprocess
import sys
from pathlib import Path

requirements = Path("requirements.txt")
assert requirements.is_file(), "Repo klasöründe requirements.txt bulunamadı."

subprocess.run(
    [
        sys.executable, "-m", "uv", "pip", "install",
        "--system",
        "--python", sys.executable,
        "-r", str(requirements),
    ],
    check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-m', 'uv', 'pip', 'install', '--system', '--python', '/usr/bin/python3', '-r', 'requirements.txt'], returncode=0)

In [16]:
# Proje yapisi ve kurulum kontrolu
from pathlib import Path
import sys

repo_root = Path('/content/GROUP37_CENG313')
assert (repo_root / 'src').is_dir(), 'src klasoru bulunamadi.'
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import src
print('src import basarili')
print('src dosyalari:', [str(p.relative_to(repo_root)) for p in (repo_root / 'src').rglob('*') if p.is_file() and '__pycache__' not in str(p)])
for name in ['README.md', 'requirements.txt', '.gitignore']:
    p = repo_root / name
    print('\n---', name, '---')
    print(p.read_text()[:12000] if p.exists() else 'Dosya yok')

src import basarili
src dosyalari: ['src/models.py', 'src/data.py', 'src/evaluation.py', 'src/_init_.py']

--- README.md ---
# image-matching-challenge-2025
"https://www.kaggle.com/competitions/image-matching-challenge-2025"

Ayşegül Pınar Yılmaz 
23118080004
Cherrylin

Kayra Orbay
23118080017 
KayraOrbay


--- requirements.txt ---


--- .gitignore ---
# Datasets and downloaded archives
data/
datasets/
train_labels.csv
train_thresholds.csv
sample_submission.csv
*.zip

# Model weights
weights/
checkpoints/
*.pt
*.pth
*.ckpt
*.h5
*.onnx
*.safetensors

# Credentials
.env
.env.*
!.env.example
secrets/
kaggle.json
credentials.json
service-account*.json
*.pem
*.key

# Python and notebook temporary files
.venv/
__pycache__/
*.pyc
.ipynb_checkpoints/



In [17]:
# Gercek proje modullerini kontrol et
import importlib
for module_name in ['src.data', 'src.models', 'src.evaluation']:
    module = importlib.import_module(module_name)
    print(module_name, 'OK:', module.__file__)
    print(Path(module.__file__).read_text()[:8000] or '(Dosya bos)')

# Veri, model agirliklari ve erisim bilgilerini Git takibinden haric tut
ignore_path = repo_root / '.gitignore'
existing = ignore_path.read_text() if ignore_path.exists() else ''
rules = ['data/', 'datasets/', 'weights/', 'checkpoints/', '*.pt', '*.pth', '*.ckpt', '*.h5', '*.onnx', '*.safetensors', '*.zip', '.env', '.env.*', '!.env.example', 'secrets/', 'kaggle.json', 'credentials.json', 'service-account*.json', '*.pem', '*.key', '__pycache__/', '*.pyc', '.ipynb_checkpoints/']
missing = [rule for rule in rules if rule not in existing.splitlines()]
if missing:
    ignore_path.write_text(existing + ('\n' if existing and not existing.endswith('\n') else '') + '\n'.join(missing) + '\n')
print('\n.gitignore hazir:', ignore_path)
print('Veri klasorleri:', [str(p) for p in [repo_root / 'data', Path('/content/data'), Path('/content/drive/MyDrive')] if p.exists()])

src.data OK: /content/GROUP37_CENG313/src/data.py
"""Read Image Matching Challenge 2025 training labels."""

import csv
from pathlib import Path


def read_train_labels(path):
    """Return label records after checking the required CSV columns."""
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {
            "dataset", "scene", "image", "rotation_matrix", "translation_vector"
        }
        if not required.issubset(reader.fieldnames or []):
            raise ValueError("Beklenen train_labels.csv sütunları eksik.")
        rows = list(reader)
    if not rows:
        raise ValueError("Etiket dosyası boş.")
    return rows

src.models OK: /content/GROUP37_CENG313/src/models.py
(Dosya bos)
src.evaluation OK: /content/GROUP37_CENG313/src/evaluation.py
(Dosya bos)

.gitignore hazir: /content/GROUP37_CENG313/.gitignore
Veri klasorleri: ['/content/GROUP37_CENG313/data']


In [19]:
# Kaggle train_labels.csv dosyasini oku; ek paket gerektirmez.
from pathlib import Path
import importlib
import sys
from google.colab import files

repo_root = Path('/content/GROUP37_CENG313')
assert (repo_root / 'src').is_dir(), 'Once repo klonlama ve kurulum hucrelerini calistirin.'
loader_source = '''import csv
from pathlib import Path

def read_train_labels(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        required = {"dataset", "scene", "image", "rotation_matrix", "translation_vector"}
        if not required.issubset(reader.fieldnames or []):
            raise ValueError("Beklenen train_labels.csv sutunlari eksik.")
        rows = list(reader)
    if not rows:
        raise ValueError("Etiket dosyasi bos.")
    return rows
'''
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import src.data
importlib.reload(src.data)

data_dir = repo_root / 'data'
data_dir.mkdir(exist_ok=True)
labels_path = data_dir / 'train_labels.csv'
if not labels_path.exists():
    print('Kaggle sitesinden indirilen train_labels.csv dosyasini secin.')
    uploaded = files.upload()
    if 'train_labels.csv' not in uploaded:
        raise ValueError('train_labels.csv secilmedi.')
    labels_path.write_bytes(uploaded['train_labels.csv'])

rows = src.data.read_train_labels(labels_path)
print('Veri okuma basarili. Satir sayisi:', len(rows))
print('Sutunlar:', list(rows[0]))
print('Ilk kayit:', rows[0])
print('Veri kumesi sayisi:', len({row['dataset'] for row in rows}))

ValueError: Beklenen train_labels.csv sütunları eksik.

In [ ]:
import pandas as pd

labels = pd.DataFrame(rows)

images_per_dataset = labels.groupby("dataset").size().sort_values(ascending=False)
images_per_scene = labels.groupby(["dataset", "scene"]).size().sort_values(ascending=False)

labels_check = labels.replace(r"^\s*$", pd.NA, regex=True)
labels_check = labels_check.replace(["NaN", "nan", "NULL", "null"], pd.NA)

missing = labels_check.isna()

print("Data Shape:", labels.shape)
print("Scene Number:", len(images_per_scene))

display(pd.DataFrame({
    "Type": labels.dtypes.astype(str),
    "Missing Value": missing.sum()
}))

print("Duplicates:", labels.duplicated().sum())
duplicate_rows = labels.loc[labels.duplicated(keep=False)]
if not duplicate_rows.empty:
    display(duplicate_rows.head(10))
display(images_per_dataset.rename("Kayıt sayısı").to_frame())